In [ ]:
import pandas as pd
import seaborn as sb
import os
import scipy.stats as stats
from matplotlib.colors import to_rgb
import numpy as np
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
plt.style.use('paper_style.mplstyle')

In [ ]:
palette2={'Non-tagged vs Non-tagged':'#00ABC8',
          'Tagged vs Non-tagged':'#FFC40C',
          'Tagged vs Tagged':'#F37243'}

def load_pairwise_df_all(bin_size): ### get the pairwise correlations for whole time. 
    '''
    bin_size: dataframes to load based on bin size in frame
    '''
    path=f'/Volumes/AM_SSD1/Spont2P/Analysis/correlation/pairwise_dataframes_reg_tot/0_lag/{bin_size}_bin'
    df_pre = pd.read_csv(os.path.join(path,'pairwise_corr_df_pre.csv'),index_col=0)
    df_post = pd.read_csv(os.path.join(path, 'pairwise_corr_df_post.csv'),index_col=0)

    return df_pre,df_post

def fisher_z_transform(r):
    """
    Apply Fisher z-transform to a correlation coefficient (r).
    
    Parameters:
        r (float or ndarray): Pearson or Spearman correlation coefficient(s).
        
    Returns:
        float or ndarray: Fisher z-transformed value(s).
    """
    return 0.5 * np.log((1 + r) / (1 - r))

def change_corr_bins_rest(bins):
    '''
    bins: list or array of bin sizes
    drop any non-firing cells 

    '''
    df_all = pd.DataFrame()
    for b in bins:
        df_pre0,df_post0 = load_pairwise_df_all(b)
        # Create a mask to check where 999 appears in either df_pre0 or df_post0
        mask_999 = (df_pre0 == 999) | (df_post0 == 999)
        # Get the indices of the rows with 999 in either dataframe
        rows_with_999 = mask_999.any(axis=1)
        # Remove these rows from df_cleaned
        df_post= df_post0[~rows_with_999]
        df_pre= df_pre0[~rows_with_999]
        abs_change = np.abs(df_post['Spearmans R']-df_pre['Spearmans R'])
        change = df_post['Spearmans R']-df_pre['Spearmans R']
        Z_change =df_post['ZScored Spearmans R']-df_pre['ZScored Spearmans R']
        Z_abs=np.abs(df_post['ZScored Spearmans R']-df_pre['ZScored Spearmans R'])
        df = df_pre.copy()
        df = df.drop(['Session','ZScored Spearmans R','pvals'],axis=1)
        df['deltaR(abs)'] = abs_change
        df['deltaR']=change
        df['deltaZScored']=Z_change
        df['absZ']=Z_abs
        df['Bin Size']=b*1/30
        df_all=pd.concat([df_all,df],ignore_index=False)
    return df_all

def filter_by_pval(df_pre, df_post, p_thresh=0.05):
    """
    Filters rows where p-value is below threshold in both df_pre and df_post.
    """
    mask = (df_pre['pvals'] < p_thresh) & (df_post['pvals'] < p_thresh)
    
    df_pre_filtered = df_pre[mask].copy()
    df_post_filtered = df_post[mask].copy()
    
    return df_pre_filtered, df_post_filtered

def filter_single_df_by_pval(df, p_thresh=0.05):
    """
    Returns rows where p-value is below the given threshold.
    """
    return df[df['pvals'] < p_thresh].copy()

def zscore_change_rest(df_change):
    
    
    
    
    
    '''
    drop any non-firing cells 
    '''
    bins = np.unique(df_change['Bin Size'])
    zscored_all=[]
    zscored_persession=[]
    zabs_all=[]
    for b in bins: #within bin sizes
        df_b = df_change.loc[df_change['Bin Size']==b]
        zscored=[]
        fisherz=[]
        zscored_session=[]
        zabs=[]
        for ani in df_b.Animal.unique(): #within each animal
            df_a = df_b.loc[df_b.Animal==ani]
            zscored.extend(stats.zscore(df_a['deltaR'])) #ZScore the change in coefficient
            zscored_session.extend(df_a['deltaZScored'])
            fisherz.extend(fisher_z_transform(df_a['deltaR']))
            zabs.extend(df_a['absZ'])
        zscored_all.extend(zscored)
        zscored_persession.extend(zscored_session)
        zabs_all.extend(zabs)
    df_change.drop(['deltaZScored','absZ'],axis=1,inplace=True) #ZScore the change
    df_change['deltaR_Zscore']=zscored_all 
    df_change['delta_z_R']=zscored_persession #ZScore within session first.
    df_change['|deltaR|']=zabs_all
    return df_change

In [ ]:
bins = [4,8,10,16,20,25,32,50,64,100,125]
df_all=pd.DataFrame()
for b in bins:
    df_pre,df_post = load_pairwise_df_all(b)
    df = pd.concat([df_pre,df_post],ignore_index=True)
    df['Bin Size (s)'] = np.round(b*1/30,decimals=3)
    df_all = pd.concat([df_all,df],ignore_index=True)
df_all=df_all.dropna()

delta_R_allbins=change_corr_bins_rest(bins) # change dataframe 
delta_R_allbins = zscore_change_rest(delta_R_allbins)  # zscored Change dataframe

In [ ]:
delta = delta_R_allbins.loc[delta_R_allbins['Bin Size']==125/30]

In [ ]:
df_single_bin = df_all.loc[(df_all['Bin Size (s)']==4.167)&(df_all['Group']=='FC')].copy()

# plot dist 

In [ ]:
g=sb.displot(data=df_single_bin,col='Session',x='Spearmans R',kind='kde',hue='Pair Group',common_norm=False,palette=palette2,height=5,aspect=.8,legend=False)


In [ ]:
pvals=[0.2,0.1,0.05,0.01,.001,.0001,.00001,.000001]
df_long=[]
for pval in pvals:
    df_pre,df_post= load_pairwise_df_all(50)
    df_pre['id']=np.arange(0,len(df_pre))
    df_post['id']=np.arange(0,len(df_post))

    # df1=filter_single_df_by_zscore(df_pre,z_thresh=2)
    # df2=filter_single_df_by_zscore(df_post,z_thresh=2)
    df1=filter_single_df_by_pval(df_pre,p_thresh=pval)
    df2=filter_single_df_by_pval(df_post,p_thresh=pval)
    
    df_thr = pd.concat([df1,df2],ignore_index=True)
    df_thr = df_thr[~df_thr.eq(999).any(axis=1)] 
    
    df = df_thr.loc[df_thr['Spearmans R']>0]
    # df = df.groupby(['Group','Animal','Pair Group','Session']).mean(['Spearmans R']).reset_index()
    df['thr']=pval
    df_long.append(df)
    print(df.loc[df['Group']=='FC'].shape)
    print(df.loc[df['Group']=='HC'].shape)

thr_dist=pd.concat(df_long,axis=0)
g=sb.displot(data=thr_dist,col='Session',x='Spearmans R',kind='kde',hue='Pair Group',common_norm=False,palette=palette2,height=5,aspect=.8,legend=False)


In [ ]:
df_single_bin['log_pval'] = -np.log10(df_single_bin['pvals'])
significance_threshold = 0.05
df_single_bin['significant'] =  df_single_bin['pvals'] < significance_threshold

g=sb.relplot(data=df_single_bin,col='Session',x='Spearmans R',y='log_pval',kind='scatter',hue='significant',palette=['k','r'],height=5,aspect=.8,legend=False,)


In [ ]:
df_means = df_single_bin.groupby(['Group','Pair Group','Session']).mean(['Spearmans R']).reset_index()
df_means

In [ ]:
df_sem = (
    df_single_bin
    .groupby(['Group', 'Pair Group', 'Session'])['Spearmans R']
    .sem()
    .reset_index(name='SEM')
)

In [ ]:
baseline = df_means[df_means['Session']=='Baseline']
means1=[baseline['Spearmans R'].loc[baseline['Pair Group']==p].values[0] for p in np.unique(baseline['Pair Group'])]
post = df_means[df_means['Session']=='Post']
means2=[post['Spearmans R'].loc[post['Pair Group']==p].values[0] for p in np.unique(post['Pair Group'])]
baseline_sem = df_sem[df_sem['Session'] == 'Baseline']['SEM'].values
post_sem = df_sem[df_sem['Session'] == 'Post']['SEM'].values

fig,ax=plt.subplots(nrows=1,ncols=2,figsize=(8,3))
sb.histplot(data= df_single_bin[df_single_bin['Session']=='Baseline'],x='Spearmans R',hue='significant',common_norm=False,palette=['.5','r'],ax=ax[0],legend=False)
sb.histplot(data= df_single_bin[df_single_bin['Session']=='Post'],x='Spearmans R',hue='significant',common_norm=False,palette=['.5','r'],ax=ax[1])
for i, (m1, s1, m2, s2) in enumerate(zip(means1, baseline_sem, means2, post_sem)):
    color = to_rgb(list(palette2.values())[i])

    # Mean lines
    ax[0].axvline(m1, linestyle='-', linewidth=1, color=color)
    ax[1].axvline(m2, linestyle='-', linewidth=1, color=color)

    # Shaded SEM regions
    ax[0].axvspan(m1 - s1, m1 + s1, color=color, alpha=0.2)
    ax[1].axvspan(m2 - s2, m2 + s2, color=color, alpha=0.2)

sb.despine()
ax[0].set_xlim([-.3,.4])
ax[1].set_xlim([-.3,.4])
plt.tight_layout()
#plt.savefig('/Users/amonast/BOSTON UNIVERSITY Dropbox/Amy Monasterio/Manuscripts/Engram2P/Figures/RevisionFigures/Figure4_Supp5/dist_v_significant.svg',transparent=True)
plt.savefig('SuppFig6A_1.svg',transparent=True)

fig,ax=plt.subplots(nrows=1,ncols=2,figsize=(8,3))
sb.histplot(data= df_single_bin[df_single_bin['Session']=='Baseline'],x='Spearmans R',hue='significant',common_norm=False,palette=['.5','r'],ax=ax[0],legend=False,alpha=.3)
sb.histplot(data= df_single_bin[df_single_bin['Session']=='Post'],x='Spearmans R',hue='significant',common_norm=False,palette=['.5','r'],ax=ax[1],legend=False,alpha=.3)
# Plot means and SEM shaded areas
for i, (m1, s1, m2, s2) in enumerate(zip(means1, baseline_sem, means2, post_sem)):
    color = to_rgb(list(palette2.values())[i])

    # Mean lines
    ax[0].axvline(m1, linestyle='-', linewidth=3, color=color)
    ax[1].axvline(m2, linestyle='-', linewidth=3, color=color)

    # Shaded SEM regions
    ax[0].axvspan(m1 - s1, m1 + s1, color=color, alpha=0.2)
    ax[1].axvspan(m2 - s2, m2 + s2, color=color, alpha=0.2)

plt.tight_layout()
ax[0].set_xlim(-0.01,0.013)
ax[1].set_xlim(-0.01,0.013)
#plt.savefig('/Users/amonast/BOSTON UNIVERSITY Dropbox/Amy Monasterio/Manuscripts/Engram2P/Figures/RevisionFigures/Figure4_Supp5/dist_v_significant_zoom.svg',transparent=True)
plt.savefig('SuppFig6A_2.svg',transparent=True)